# 模型分数分箱

按照 [模型分数分箱操作指南](./模型分数分箱操作指南.md) 的字段口径，实现等频分箱、指标计算、合并相邻箱、阈值曲线和策略阈值确定。

## 0. 环境配置

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.stats import chi2_contingency
import statsmodels.api as sm

# 显示设置
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path('res')

# 时间切分配置：后续可按业务确认后调整
TRAIN_END_MONTH = '2026-03'
OOT_START_MONTH = '2026-04'

# 模型分方向：当前方法论按高分高风险处理
HIGH_SCORE_HIGH_RISK = True


## 1. 加载数据

In [ ]:
# ============================================================
# 1. 加载数据
# ============================================================

def clean_columns(frame):
    """清理 UTF-8 BOM 以及少数 CSV 头部乱码。"""
    frame = frame.copy()
    frame.columns = [str(c).lstrip('\ufeff').lstrip('ï»¿') for c in frame.columns]
    return frame


def read_csv_clean(path, **kwargs):
    frame = pd.read_csv(path, low_memory=False, **kwargs)
    return clean_columns(frame)


# 主表：样本表
sample = read_csv_clean(DATA_DIR / 'sample.csv')
print(f"sample: {sample.shape}")

# 申请信息表（已按 sample 对齐）
app = read_csv_clean(DATA_DIR / 'application_info.csv')
print(f"app: {app.shape}")

# 模型分表
mlt_score = read_csv_clean(DATA_DIR / 'aus_old_risk_bid_mltmodel_v1_2_20260325_lgb_score.csv')
apply_score = read_csv_clean(DATA_DIR / 'aus_old_risk_apply_appmodel_v20260318_v1_2_lgb_score.csv')
txn_score = read_csv_clean(DATA_DIR / 'aus_old_risk_bid_submodel_v20260323_v1_2_txn_lgb_score.csv')
print(f"mlt_score: {mlt_score.shape}")
print(f"apply_score: {apply_score.shape}")
print(f"txn_score: {txn_score.shape}")


## 2. 拼接主表与数据校验

In [ ]:
# ============================================================
# 2. 拼接主表（与 scr/application_info_extract.sql 口径一致）
# ============================================================

# 以 sample 为底表，左连接各表
df = sample.merge(app, on=['application_id', 'user_id'], how='left')

# 模型分：去重后保留 score 列，重命名
# mlt_score 和 apply_score 当前各有少量重复 application_id，先取第一条
mlt_col = 'aus_old_risk_bid_mltmodel_v1_2_v20260325_lgb_score'
apply_col = 'aus_old_risk_apply_appmodel_v20260318_v1_2_lgb_score'

mlt_dedup = mlt_score.drop_duplicates(subset='application_id', keep='first')
apply_dedup = apply_score.drop_duplicates(subset='application_id', keep='first')
print(f"mlt_score 去重: {mlt_score.shape[0]} -> {mlt_dedup.shape[0]}")
print(f"apply_score 去重: {apply_score.shape[0]} -> {apply_dedup.shape[0]}")

df = df.merge(
    mlt_dedup[['application_id', mlt_col]],
    on='application_id', how='left'
).rename(columns={mlt_col: 'score_mlt'})

df = df.merge(
    apply_dedup[['application_id', apply_col]],
    on='application_id', how='left'
).rename(columns={apply_col: 'score_apply'})

# 交易特征子模型：保留 score + 交易特征
txn_feature_cols = [
    c for c in txn_score.columns
    if c not in (
        'application_id', 'user_id', 'sample_datetime',
        'feature_error', 'send_time',
        'transaction_date_max', 'balance_date_max'
    )
]
txn_dedup = txn_score.drop_duplicates(subset='application_id', keep='first')
print(f"txn_score 去重: {txn_score.shape[0]} -> {txn_dedup.shape[0]}")

df = df.merge(
    txn_dedup[['application_id'] + txn_feature_cols],
    on='application_id', how='left'
)

# 只保留分箱需要的字段
keep_cols = [
    'application_id', 'user_id',
    'application_time', 'application_date', 'application_month',
    'score_mlt', 'score_apply',
    *txn_feature_cols,
    'duedate_1m_30', 'duedate_3m_30',
    'principal', 'estimate_principal_remaining_mob1',
    'estimate_principal_remaining_mob3',
    'dpd_days_ever_mob1', 'dpd_days_ever_mob3',
    'application_status', 'assessment_status', 'status',
    'LTI', 'PTI', 'NSTI',
    'application_tag', 'user_tag', 'loan_tag',
]

keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

# 基础类型转换
date_cols = ['application_time', 'application_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

numeric_cols = [
    'score_mlt', 'score_apply',
    'duedate_1m_30', 'duedate_3m_30',
    'principal', 'estimate_principal_remaining_mob1',
    'estimate_principal_remaining_mob3',
    'dpd_days_ever_mob1', 'dpd_days_ever_mob3',
    'LTI', 'PTI', 'NSTI',
]
numeric_cols = [c for c in numeric_cols if c in df.columns]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"df: {df.shape}")
print(f"columns ({len(df.columns)}): {df.columns.tolist()}")
print(f"\n缺失率概览:\n{df.isnull().mean().round(4).sort_values(ascending=False).head(20)}")


In [ ]:
# ============================================================
# 2.1 数据校验
# ============================================================

required_cols = [
    'application_id', 'user_id', 'application_time', 'application_month',
    'score_mlt', 'score_apply',
    'duedate_1m_30', 'duedate_3m_30',
    'principal', 'estimate_principal_remaining_mob1',
    'estimate_principal_remaining_mob3',
    'dpd_days_ever_mob1', 'dpd_days_ever_mob3',
]
missing_required_cols = [c for c in required_cols if c not in df.columns]
if missing_required_cols:
    raise ValueError(f"关键字段缺失: {missing_required_cols}")

data_quality = pd.DataFrame({
    'column': df.columns,
    'dtype': [df[c].dtype for c in df.columns],
    'missing_cnt': [df[c].isna().sum() for c in df.columns],
    'missing_rate': [df[c].isna().mean() for c in df.columns],
    'nunique': [df[c].nunique(dropna=True) for c in df.columns],
})

key_checks = {
    'sample_rows': len(sample),
    'df_rows': len(df),
    'sample_application_id_dup': int(sample['application_id'].duplicated().sum()),
    'df_application_id_dup': int(df['application_id'].duplicated().sum()),
    'mlt_score_application_id_dup': int(mlt_score['application_id'].duplicated().sum()),
    'apply_score_application_id_dup': int(apply_score['application_id'].duplicated().sum()),
    'txn_score_application_id_dup': int(txn_score['application_id'].duplicated().sum()),
    'application_month_min': df['application_month'].min(),
    'application_month_max': df['application_month'].max(),
    'application_month_nunique': df['application_month'].nunique(dropna=True),
}

display(pd.Series(key_checks, name='value'))
display(data_quality.sort_values('missing_rate', ascending=False).head(30))


## 3. 样本切分

In [ ]:
# ============================================================
# 3. 样本切分
# ============================================================

application_month_for_split = df['application_month'].astype('string')
train_mask = application_month_for_split.notna() & application_month_for_split.le(TRAIN_END_MONTH)
oot_mask = application_month_for_split.notna() & application_month_for_split.ge(OOT_START_MONTH)

df['sample_group'] = np.select(
    [
        train_mask.to_numpy(dtype=bool, na_value=False),
        oot_mask.to_numpy(dtype=bool, na_value=False),
    ],
    ['train', 'oot'],
    default='gap_or_unknown'
)

train_df = df.loc[df['sample_group'].eq('train')].copy()
oot_df = df.loc[df['sample_group'].eq('oot')].copy()

split_summary = (
    df.groupby('sample_group', dropna=False)
      .agg(
          n=('application_id', 'count'),
          application_id_nunique=('application_id', 'nunique'),
          month_min=('application_month', 'min'),
          month_max=('application_month', 'max'),
          score_mlt_missing_rate=('score_mlt', lambda s: s.isna().mean()),
          m1_mature=('duedate_1m_30', lambda s: s.isin([0, 1]).sum()),
          m3_mature=('duedate_3m_30', lambda s: s.isin([0, 1]).sum()),
      )
      .reset_index()
)

display(split_summary)
print(f"train_df: {train_df.shape}")
print(f"oot_df: {oot_df.shape}")


## 4. 指标计算引擎

In [ ]:
# ============================================================
# 4. 指标计算引擎：通用辅助函数
# ============================================================

def safe_div(num, den):
    """支持标量和 Series 的安全除法，分母为 0 时返回 NaN。"""
    num_arr = np.asarray(num, dtype='float64')
    den_arr = np.asarray(den, dtype='float64')
    result = np.full(np.broadcast(num_arr, den_arr).shape, np.nan, dtype='float64')
    np.divide(num_arr, den_arr, out=result, where=den_arr != 0)
    if np.ndim(result) == 0:
        return float(result)
    if isinstance(num, pd.Series):
        return pd.Series(result, index=num.index)
    if isinstance(den, pd.Series):
        return pd.Series(result, index=den.index)
    return result


def wilson_ci(numerator, denominator, alpha=0.05):
    """Wilson score interval for a binomial proportion."""
    numerator = pd.Series(numerator, dtype='float64')
    denominator = pd.Series(denominator, dtype='float64')
    z = norm.ppf(1 - alpha / 2)
    p = safe_div(numerator, denominator)

    lower = pd.Series(np.nan, index=numerator.index, dtype='float64')
    upper = pd.Series(np.nan, index=numerator.index, dtype='float64')
    valid = denominator.gt(0)

    denom = 1 + z ** 2 / denominator.loc[valid]
    center = (p.loc[valid] + z ** 2 / (2 * denominator.loc[valid])) / denom
    margin = (
        z
        * np.sqrt((p.loc[valid] * (1 - p.loc[valid]) + z ** 2 / (4 * denominator.loc[valid])) / denominator.loc[valid])
        / denom
    )
    lower.loc[valid] = (center - margin).clip(lower=0)
    upper.loc[valid] = (center + margin).clip(upper=1)
    return lower, upper


def require_columns(frame, columns, context='dataframe'):
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise ValueError(f"{context} 缺少必要字段: {missing}")


In [ ]:
# ============================================================
# 4.1 分箱指标计算函数
# ============================================================

def calc_bin_stats(data, bin_col, score_col=None, id_col='application_id'):
    """
    按分箱计算完整风险指标。

    口径：
    - 笔数 1M30+/3M30+ 成熟样本：duedate_1m_30 / duedate_3m_30 in [0, 1]
    - 金额 1M30+/3M30+ 成熟本金：dpd_days_ever_mob1 / mob3 非空对应的 principal
    - 金额逾期本金：成熟且 dpd_days_ever_mob >= 30 的 estimate_principal_remaining_mob
    - 累计指标默认按 bin_order 从低风险端向高风险端累计
    """
    required = [
        id_col, bin_col,
        'duedate_1m_30', 'duedate_3m_30',
        'principal', 'estimate_principal_remaining_mob1',
        'estimate_principal_remaining_mob3',
        'dpd_days_ever_mob1', 'dpd_days_ever_mob3',
    ]
    if score_col is not None:
        required.append(score_col)
    require_columns(data, required, context='calc_bin_stats')

    work = data.copy()
    for col in [
        'duedate_1m_30', 'duedate_3m_30',
        'principal', 'estimate_principal_remaining_mob1',
        'estimate_principal_remaining_mob3',
        'dpd_days_ever_mob1', 'dpd_days_ever_mob3',
    ]:
        work[col] = pd.to_numeric(work[col], errors='coerce')

    work['_principal_fill0'] = work['principal'].fillna(0)
    work['_m1_mature_cnt_flag'] = work['duedate_1m_30'].isin([0, 1])
    work['_m1_bad_cnt_flag'] = work['duedate_1m_30'].eq(1)
    work['_m3_mature_cnt_flag'] = work['duedate_3m_30'].isin([0, 1])
    work['_m3_bad_cnt_flag'] = work['duedate_3m_30'].eq(1)

    work['_m1_mature_amt_flag'] = work['dpd_days_ever_mob1'].notna()
    work['_m1_bad_amt_flag'] = work['_m1_mature_amt_flag'] & work['dpd_days_ever_mob1'].ge(30)
    work['_m1_amt_exposure_value'] = np.where(work['_m1_mature_amt_flag'], work['_principal_fill0'], 0)
    work['_m1_amt_bad_value'] = np.where(
        work['_m1_bad_amt_flag'],
        work['estimate_principal_remaining_mob1'].fillna(0),
        0,
    )

    work['_m3_mature_amt_flag'] = work['dpd_days_ever_mob3'].notna()
    work['_m3_bad_amt_flag'] = work['_m3_mature_amt_flag'] & work['dpd_days_ever_mob3'].ge(30)
    work['_m3_amt_exposure_value'] = np.where(work['_m3_mature_amt_flag'], work['_principal_fill0'], 0)
    work['_m3_amt_bad_value'] = np.where(
        work['_m3_bad_amt_flag'],
        work['estimate_principal_remaining_mob3'].fillna(0),
        0,
    )

    group_cols = [bin_col]
    if 'bin_order' in work.columns and bin_col != 'bin_order':
        group_cols.append('bin_order')

    agg_dict = {
        'n': (id_col, 'count'),
        'application_id_nunique': (id_col, 'nunique'),
        'principal_amt': ('_principal_fill0', 'sum'),
        '1m30p_cnt_mature': ('_m1_mature_cnt_flag', 'sum'),
        '1m30p_cnt_bad': ('_m1_bad_cnt_flag', 'sum'),
        '3m30p_cnt_mature': ('_m3_mature_cnt_flag', 'sum'),
        '3m30p_cnt_bad': ('_m3_bad_cnt_flag', 'sum'),
        '1m30p_amt_exposure': ('_m1_amt_exposure_value', 'sum'),
        '1m30p_amt_bad': ('_m1_amt_bad_value', 'sum'),
        '1m30p_amt_bad_cnt': ('_m1_bad_amt_flag', 'sum'),
        '3m30p_amt_exposure': ('_m3_amt_exposure_value', 'sum'),
        '3m30p_amt_bad': ('_m3_amt_bad_value', 'sum'),
        '3m30p_amt_bad_cnt': ('_m3_bad_amt_flag', 'sum'),
    }
    if score_col is not None:
        agg_dict.update({
            'score_min': (score_col, 'min'),
            'score_max': (score_col, 'max'),
            'score_mean': (score_col, 'mean'),
        })

    bin_stats = work.groupby(group_cols, dropna=False, observed=True).agg(**agg_dict).reset_index()
    if 'bin_order' not in bin_stats.columns:
        bin_stats['bin_order'] = np.arange(1, len(bin_stats) + 1)
    bin_stats = bin_stats.sort_values('bin_order').reset_index(drop=True)

    total_n = bin_stats['n'].sum()
    bin_stats['total_n'] = total_n
    bin_stats['sample_pct_num'] = bin_stats['n']
    bin_stats['sample_pct_den'] = total_n
    bin_stats['sample_pct'] = safe_div(bin_stats['sample_pct_num'], bin_stats['sample_pct_den'])

    # 笔数口径
    for prefix in ['1m30p', '3m30p']:
        bin_stats[f'{prefix}_cnt_good'] = bin_stats[f'{prefix}_cnt_mature'] - bin_stats[f'{prefix}_cnt_bad']
        bin_stats[f'{prefix}_cnt_bad_rate_num'] = bin_stats[f'{prefix}_cnt_bad']
        bin_stats[f'{prefix}_cnt_bad_rate_den'] = bin_stats[f'{prefix}_cnt_mature']
        bin_stats[f'{prefix}_cnt_bad_rate'] = safe_div(
            bin_stats[f'{prefix}_cnt_bad_rate_num'],
            bin_stats[f'{prefix}_cnt_bad_rate_den'],
        )

    # 金额口径
    for prefix in ['1m30p', '3m30p']:
        bin_stats[f'{prefix}_amt_bad_rate_num'] = bin_stats[f'{prefix}_amt_bad']
        bin_stats[f'{prefix}_amt_bad_rate_den'] = bin_stats[f'{prefix}_amt_exposure']
        bin_stats[f'{prefix}_amt_bad_rate'] = safe_div(
            bin_stats[f'{prefix}_amt_bad_rate_num'],
            bin_stats[f'{prefix}_amt_bad_rate_den'],
        )

    # Lift
    overall_rates = {}
    for prefix in ['1m30p', '3m30p']:
        overall_rates[f'{prefix}_cnt'] = safe_div(
            bin_stats[f'{prefix}_cnt_bad'].sum(),
            bin_stats[f'{prefix}_cnt_mature'].sum(),
        )
        bin_stats[f'{prefix}_cnt_lift_num'] = bin_stats[f'{prefix}_cnt_bad_rate']
        bin_stats[f'{prefix}_cnt_lift_den'] = overall_rates[f'{prefix}_cnt']
        bin_stats[f'{prefix}_cnt_lift'] = safe_div(
            bin_stats[f'{prefix}_cnt_lift_num'],
            bin_stats[f'{prefix}_cnt_lift_den'],
        )

        overall_rates[f'{prefix}_amt'] = safe_div(
            bin_stats[f'{prefix}_amt_bad'].sum(),
            bin_stats[f'{prefix}_amt_exposure'].sum(),
        )
        bin_stats[f'{prefix}_amt_lift_num'] = bin_stats[f'{prefix}_amt_bad_rate']
        bin_stats[f'{prefix}_amt_lift_den'] = overall_rates[f'{prefix}_amt']
        bin_stats[f'{prefix}_amt_lift'] = safe_div(
            bin_stats[f'{prefix}_amt_lift_num'],
            bin_stats[f'{prefix}_amt_lift_den'],
        )

    # 笔数逾期率标准误和 Wilson 置信区间
    for prefix in ['1m30p', '3m30p']:
        rate_col = f'{prefix}_cnt_bad_rate'
        mature_col = f'{prefix}_cnt_mature'
        bin_stats[f'{prefix}_cnt_bad_rate_se'] = np.sqrt(
            bin_stats[rate_col]
            * (1 - bin_stats[rate_col])
            / bin_stats[mature_col].where(bin_stats[mature_col].gt(0), np.nan)
        )
        lower, upper = wilson_ci(bin_stats[f'{prefix}_cnt_bad'], bin_stats[mature_col])
        bin_stats[f'{prefix}_cnt_bad_rate_ci_lower'] = lower
        bin_stats[f'{prefix}_cnt_bad_rate_ci_upper'] = upper

    # 累计指标：从低风险端向高风险端逐箱累计
    bin_stats['cum_n'] = bin_stats['n'].cumsum()
    bin_stats['cum_principal'] = bin_stats['principal_amt'].cumsum()
    bin_stats['cum_pass_rate_num'] = bin_stats['cum_n']
    bin_stats['cum_pass_rate_den'] = total_n
    bin_stats['cum_pass_rate'] = safe_div(bin_stats['cum_pass_rate_num'], bin_stats['cum_pass_rate_den'])

    for prefix in ['1m30p', '3m30p']:
        bin_stats[f'cum_{prefix}_cnt_mature'] = bin_stats[f'{prefix}_cnt_mature'].cumsum()
        bin_stats[f'cum_{prefix}_cnt_bad'] = bin_stats[f'{prefix}_cnt_bad'].cumsum()
        bin_stats[f'cum_{prefix}_cnt_bad_rate_num'] = bin_stats[f'cum_{prefix}_cnt_bad']
        bin_stats[f'cum_{prefix}_cnt_bad_rate_den'] = bin_stats[f'cum_{prefix}_cnt_mature']
        bin_stats[f'cum_{prefix}_cnt_bad_rate'] = safe_div(
            bin_stats[f'cum_{prefix}_cnt_bad_rate_num'],
            bin_stats[f'cum_{prefix}_cnt_bad_rate_den'],
        )

        bin_stats[f'cum_{prefix}_amt_exposure'] = bin_stats[f'{prefix}_amt_exposure'].cumsum()
        bin_stats[f'cum_{prefix}_amt_bad'] = bin_stats[f'{prefix}_amt_bad'].cumsum()
        bin_stats[f'cum_{prefix}_amt_bad_cnt'] = bin_stats[f'{prefix}_amt_bad_cnt'].cumsum()
        bin_stats[f'cum_{prefix}_amt_bad_rate_num'] = bin_stats[f'cum_{prefix}_amt_bad']
        bin_stats[f'cum_{prefix}_amt_bad_rate_den'] = bin_stats[f'cum_{prefix}_amt_exposure']
        bin_stats[f'cum_{prefix}_amt_bad_rate'] = safe_div(
            bin_stats[f'cum_{prefix}_amt_bad_rate_num'],
            bin_stats[f'cum_{prefix}_amt_bad_rate_den'],
        )

    return bin_stats


In [ ]:
# ============================================================
# 4.2 漏斗指标计算函数
# ============================================================

def calc_funnel_stats(data, group_col=None, id_col='application_id'):
    """计算申请、审批、自动/人工审批和成交漏斗。"""
    required = [id_col, 'application_status', 'assessment_status', 'status']
    require_columns(data, required, context='calc_funnel_stats')

    def one_group(g):
        apply_cnt = g[id_col].nunique()
        completed_application_cnt = g.loc[
            ~g['application_status'].isin(['0.Incomplete', '1.In Progress']),
            id_col,
        ].nunique()
        approved_application_cnt = g.loc[
            g['application_status'].astype(str).str[0].isin(['3', '4']),
            id_col,
        ].nunique()
        auto_approved_application_cnt = g.loc[
            g['application_status'].astype(str).str[0].isin(['3', '4'])
            & g['assessment_status'].astype(str).str.contains('Auto Approved', na=False),
            id_col,
        ].nunique()
        manual_approved_application_cnt = g.loc[
            g['application_status'].astype(str).str[0].isin(['3', '4'])
            & g['assessment_status'].astype(str).str.contains('Manual Approved', na=False),
            id_col,
        ].nunique()
        deal_sample_cnt = g.loc[
            g['application_status'].astype(str).str[0].isin(['3', '4'])
            & g['status'].isin(['Active_Account', 'Closed', 'Blocked']),
            id_col,
        ].nunique()

        return pd.Series({
            'apply_cnt': apply_cnt,
            'completed_application_cnt': completed_application_cnt,
            'approved_application_cnt': approved_application_cnt,
            'auto_approved_application_cnt': auto_approved_application_cnt,
            'manual_approved_application_cnt': manual_approved_application_cnt,
            'deal_sample_cnt': deal_sample_cnt,
            'completion_rate': safe_div(completed_application_cnt, apply_cnt),
            'approval_rate': safe_div(approved_application_cnt, completed_application_cnt),
            'auto_approval_rate': safe_div(auto_approved_application_cnt, completed_application_cnt),
            'manual_approval_rate': safe_div(manual_approved_application_cnt, completed_application_cnt),
            'auto_approval_share': safe_div(auto_approved_application_cnt, approved_application_cnt),
            'manual_approval_share': safe_div(manual_approved_application_cnt, approved_application_cnt),
            'deal_rate': safe_div(deal_sample_cnt, approved_application_cnt),
        })

    if group_col is None:
        return one_group(data).to_frame().T

    require_columns(data, [group_col], context='calc_funnel_stats group_col')
    rows = []
    for group_value, group_data in data.groupby(group_col, dropna=False, observed=True):
        row = one_group(group_data)
        row[group_col] = group_value
        rows.append(row)

    if not rows:
        return pd.DataFrame(columns=[group_col])
    return pd.DataFrame(rows)[[group_col] + [c for c in rows[0].index if c != group_col]].reset_index(drop=True)


metric_columns_preview = [
    'n', 'sample_pct', 'principal_amt',
    '1m30p_cnt_mature', '1m30p_cnt_bad_rate',
    '3m30p_cnt_mature', '3m30p_cnt_bad_rate',
    '1m30p_amt_exposure', '1m30p_amt_bad_rate',
    '3m30p_amt_exposure', '3m30p_amt_bad_rate',
    '1m30p_cnt_lift', '3m30p_cnt_lift',
    'cum_pass_rate', 'cum_1m30p_cnt_bad_rate', 'cum_3m30p_cnt_bad_rate',
]

print('指标计算函数已就绪：calc_bin_stats / calc_funnel_stats')


## 5. 等频初分

In [ ]:
# ============================================================
# 5. 等频初分：边界学习与复用
# ============================================================

def learn_equal_freq_edges(data, score_col, n_bins=20):
    """在训练样本上学习等频分箱边界，首尾扩展为 -inf / inf。"""
    require_columns(data, [score_col], context='learn_equal_freq_edges')
    score = pd.to_numeric(data[score_col], errors='coerce').dropna()
    if score.empty:
        raise ValueError(f"{score_col} 全为空，无法分箱")

    _, raw_edges = pd.qcut(score, q=n_bins, retbins=True, duplicates='drop')
    edges = np.asarray(raw_edges, dtype='float64')
    edges = np.unique(edges)
    if len(edges) < 2:
        raise ValueError(f"{score_col} 可用唯一值不足，无法形成有效分箱")

    edges[0] = -np.inf
    edges[-1] = np.inf
    return edges


def build_bin_edge_table(edges, bin_prefix='B'):
    """生成分箱边界配置表。"""
    rows = []
    for i in range(len(edges) - 1):
        bin_order = i + 1
        bin_label = f"{bin_prefix}{bin_order:02d}"
        rows.append({
            'bin_order': bin_order,
            'bin_label': bin_label,
            'score_left': edges[i],
            'score_right': edges[i + 1],
            'interval_rule': '(left, right]',
        })
    return pd.DataFrame(rows)


def apply_equal_freq_edges(data, score_col, edges, bin_col):
    """将已学习的边界套用到任意样本，保证 train / OOT 口径一致。"""
    require_columns(data, [score_col], context='apply_equal_freq_edges')
    out = data.copy()
    labels = list(range(1, len(edges)))
    bin_order = pd.cut(
        pd.to_numeric(out[score_col], errors='coerce'),
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=True,
    )
    out['bin_order'] = bin_order.astype('Int64')
    label_map = {i: f"B{i:02d}" for i in labels}
    out[bin_col] = out['bin_order'].map(label_map)
    return out


def make_equal_freq_bins(data, score_col, n_bins=20, bin_col='bin20'):
    """学习等频边界并返回带分箱字段的数据、边界数组和边界表。"""
    edges = learn_equal_freq_edges(data, score_col=score_col, n_bins=n_bins)
    binned = apply_equal_freq_edges(data, score_col=score_col, edges=edges, bin_col=bin_col)
    edge_table = build_bin_edge_table(edges)
    edge_table = edge_table.rename(columns={'bin_label': bin_col})
    return binned, edges, edge_table


In [ ]:
# ============================================================
# 5.1 主模型 score_mlt 20 等频初分
# ============================================================

SCORE_COL = 'score_mlt'
BIN20_COL = 'score_mlt_bin20'

train_binned_20, score_mlt_bin_edges, score_mlt_bin_edges_df = make_equal_freq_bins(
    train_df,
    score_col=SCORE_COL,
    n_bins=20,
    bin_col=BIN20_COL,
)

# 使用训练期边界套用全量和 OOT，不能在 OOT 上重新学习边界
df_binned_20 = apply_equal_freq_edges(
    df,
    score_col=SCORE_COL,
    edges=score_mlt_bin_edges,
    bin_col=BIN20_COL,
)
train_binned_20 = df_binned_20.loc[df_binned_20['sample_group'].eq('train')].copy()
oot_binned_20 = df_binned_20.loc[df_binned_20['sample_group'].eq('oot')].copy()

bin_stats_20 = calc_bin_stats(
    train_binned_20,
    bin_col=BIN20_COL,
    score_col=SCORE_COL,
)
bin_stats_20 = bin_stats_20.merge(
    score_mlt_bin_edges_df,
    on=['bin_order', BIN20_COL],
    how='left',
)

bin20_preview_cols = [
    'bin_order', BIN20_COL, 'score_left', 'score_right',
    'n', 'sample_pct', 'score_min', 'score_max',
    '1m30p_cnt_mature', '1m30p_cnt_bad_rate',
    '3m30p_cnt_mature', '3m30p_cnt_bad_rate',
    '1m30p_amt_bad_rate', '3m30p_amt_bad_rate',
    'cum_pass_rate', 'cum_1m30p_cnt_bad_rate', 'cum_3m30p_cnt_bad_rate',
]

print(f"score_mlt 等频初分实际箱数: {len(score_mlt_bin_edges) - 1}")
display(score_mlt_bin_edges_df)
display(bin_stats_20[bin20_preview_cols])
